In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score
)

# 1. Load and clean
df = pd.read_csv("workforce_attendance_sample.csv")
target = "risk_level"

df = df.drop_duplicates()
df = df.dropna(subset=[target]).reset_index(drop=True)

# Exclude target-derived fields and identifiers
drop_cols = [
    target, "risk_reason", "late_severity_band",
    "absence_category", "overtime_band",
    "manager_action_category",
    "manager_follow_up_required_flag",
    "coaching_recommended_flag",
    "employee_id", "employee_name", "record_id"
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target]

# Remove columns with no usable values
X = X.dropna(axis=1, how="all")
X = X.select_dtypes(exclude=["complex"])

# Convert datetime columns into numeric components
for col in X.select_dtypes(include=["datetime", "datetimetz"]).columns:
    X[f"{col}_year"] = X[col].dt.year
    X[f"{col}_month"] = X[col].dt.month
    X[f"{col}_day"] = X[col].dt.day
    X = X.drop(columns=col)

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Identify feature types
numeric_cols = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

# 4. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# 5. Build complete pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Train and evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))

if y.nunique() == 2:
    print("ROC-AUC:", round(
        roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4
    ))

# 7. Save model and metadata
os.makedirs("saved_models", exist_ok=True)

joblib.dump(model, "saved_models/workforce_attendance_model.pkl")
joblib.dump({
    "target": target,
    "features": X.columns.tolist(),
    "numeric_features": numeric_cols,
    "categorical_features": categorical_cols
}, "saved_models/workforce_attendance_model_metadata.pkl")

print("Workforce model and metadata saved.")

Accuracy: 0.9965
              precision    recall  f1-score   support

        High       0.99      1.00      0.99       410
         Low       1.00      1.00      1.00       177
      Medium       1.00      1.00      1.00       829

    accuracy                           1.00      1416
   macro avg       1.00      1.00      1.00      1416
weighted avg       1.00      1.00      1.00      1416

Workforce model and metadata saved.
